In [1]:
import torch
import torch.nn as nn
import math

class Lora(nn.Module):
  def __init__(self , in_f ,  out_f , rank=8 , lora_alpha = 16):
    super().__init__()
    self.rank = rank
    self.lora_alpha = lora_alpha
    self.linear = nn.Linear(in_f, out_f)
    self.linear.weight.requires_grad = False

    self.lora_A = nn.Parameter(torch.randn(rank , in_f))
    self.lora_B = nn.Parameter(torch.randn(out_f , rank))
    self.scaling = self.lora_alpha / rank

  def reset_parameters(self):

    nn.init.kaiming_uniform_(self.lora_A , a=math.sqrt(5))
    nn.init.zeros_(self.lora_B)

  def forward(self, x):
    base_output = self.linear(x)

    lora_output = (x @ self.lora_A.t() @ self.lora_B.t())

    return base_output + lora_output * self.scaling




In [2]:
import torch
import torch.nn as nn

class BaseModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # Frozen Embedding (Standard knowledge)
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lora_layer = Lora(embed_dim, hidden_dim, rank=8, lora_alpha=16)

        self.classifier = nn.Linear(hidden_dim, 2) # [0/1 probs]. 64 x 2

    def forward(self, x):
        x = self.embedding(x).mean(dim=1) # Simple pooling
        x = self.lora_layer(x)            # This uses the LoRA math we built!
        return self.classifier(x)

# x: The input tokens (e.g., [102, 45, 8]. "Server is down")

model = BaseModel(vocab_size=1000, embed_dim=128, hidden_dim=64)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Mock Data: "My database is crashing" -> Label 0 (Negative/Urgent)
inputs = torch.randint(0, 1000, (4, 10)) # 4 sentencs with 10 words each here 0,1000 specify range of rand no.
labels = torch.tensor([0, 1, 0, 1])      # Mock labels/outputs to finetune on



In [3]:
# Training Step

for epoch in range(5):
  model.train()
  optimizer.zero_grad()
  outputs = model(inputs)

  if epoch == 4:
    print("Final outputs:\n", outputs)

  loss = criterion(outputs, labels)

  loss.backward()      # PYTORCH MATH: Only calculates gradients for A and B!
  optimizer.step()     # Updates A and B; the base weights remain UNTOUCHED.

Final outputs:
 tensor([[  7.9963,  -6.4958],
        [ -9.1986,   1.8809],
        [ 16.7667,   5.8351],
        [-16.6025,   1.6475]], grad_fn=<AddmmBackward0>)


## New Section

#### Once we have trained lora , its time to save lora trained weights into a file to load whenever required (delta w)

In [4]:
# in my code we have only trained matrices where requires_grad = true
# that means only matrices A , B and the classifier

# Create a dictionary of only the trainable LoRA parameters
lora_weights = {name: param for name, param in model.named_parameters() if "lora_" in name}

# Save it to a tiny file
torch.save(lora_weights, "lora_adapter.pt")
print("Saved tiny LoRA adapter!")

Saved tiny LoRA adapter!


In [5]:
# 1. Initialize the same architecture
new_model = BaseModel(vocab_size=1000, embed_dim=128, hidden_dim=64)

# 2. Load the tiny adapter file
loaded_weights = torch.load("lora_adapter.pt")

# 3. Inject the weights into the model
new_model.load_state_dict(loaded_weights, strict=False)
# 'strict=False' is key: it tells PyTorch to ignore the missing frozen weights
print("Adapter loaded successfully!")

Adapter loaded successfully!


In [6]:
# 1. Prepare for Prediction
new_model.eval() # CRITICAL: Turns off Dropout and BatchNormalization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model.to(device)


test = torch.randint(0, 1000, (1, 10)).to(device) # [Batch=1, Length=10]

# 3. Run Inference without calculating Gradients (Saves VRAM)
with torch.no_grad():
    logits = new_model(test)
    print("Logits:\n", logits)
    probs = torch.softmax(logits, dim=1)
    prediction = torch.argmax(probs, dim=1)

# 5. Output Result
sentiment = "Positive" if prediction.item() == 1 else "Negative/Urgent"
print(f"Ticket Analysis: {sentiment} ({probs.max().item()*100:.2f}%)")



Logits:
 tensor([[9.7860, 3.2686]])
Ticket Analysis: Negative/Urgent (99.85%)
